# 03 — Distillation smoke test

Teacher: ProsusAI/finbert (110M params)
Student: backend.perception.semantic.distilled_llm.DistilledFinancialEncoder (~15M)

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

from backend.perception.semantic.distilled_llm import (
    TEACHER_MODEL,
    DistilledFinancialEncoder,
    load_teacher,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

/home/pyros05/Escritorio/lumina_project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HEADLINES = [
    "Apple beats earnings expectations by 10%.",
    "Apple beats earnings expectations by 10%, but warns of catastrophic supply chain failure.",
    "Federal Reserve cuts interest rates by 50 basis points.",
    "Tech stocks rally on AI optimism.",
    "Crude oil drops 5% on demand concerns.",
    "Healthcare sector under pressure as drug pricing reform debated.",
] * 10  # 60 headlines total

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
teacher = load_teacher(device=device)

student = DistilledFinancialEncoder().to(device)
n_params = sum(p.numel() for p in student.parameters())
print(f"Student params: {n_params / 1e6:.1f}M")
assert 8e6 < n_params < 25e6, "Student size out of expected range"

opt = torch.optim.AdamW(student.parameters(), lr=5e-5)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 69794.00it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Student params: 11.3M


## Distillation loop (3 epochs)

In [3]:
losses = []
for epoch in range(3):
    epoch_loss = 0.0
    for h in HEADLINES:
        enc = tokenizer(
            h,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            t_out = teacher(**enc).last_hidden_state[:, 0]
        s_emb, s_proj = student(enc["input_ids"], enc["attention_mask"])
        loss_mse = F.mse_loss(s_proj, t_out)
        loss_cos = 1.0 - F.cosine_similarity(s_proj, t_out).mean()
        loss = 0.5 * loss_mse + 0.5 * loss_cos
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(HEADLINES))
    print(f"epoch {epoch}: distill loss = {losses[-1]:.4f}")

epoch 0: distill loss = 0.4166
epoch 1: distill loss = 0.1821
epoch 2: distill loss = 0.1138


In [4]:
assert losses[-1] < losses[0], "Distillation did not decrease the loss"
print("PASS — distillation pipeline works.")

PASS — distillation pipeline works.
